# Real-content spoiler evaluation — no training
Review the candidate pack with the local blind-review tool first. Upload the **complete reviewed JSONL** containing both dev and test, with verified work/source groups. This notebook refuses pending labels; uncertain reviewed records are excluded from binary metrics.

Select T4. Compare RoBERTa truncation vs overlapping windows on **development only**. Select the configuration and threshold with highest development spoiler F1, freeze the selection, then score test once. These brief excerpts are not a full-review or full-video benchmark. No production-quality claim follows from a small convenience sample.

Files under /content are temporary. Download the cache ZIP after a stage; upload it on a later session to resume with identical data/model/config. Changing library versions may invalidate caches. No Drive mount or paid resources are configured.


In [ ]:
from pathlib import Path
Path("/content/product_eval.py").write_text('"""No-training, resumable HF product evaluation; run --help for CLI.\n\nOne JSONL must contain both dev and test to enforce group disjointness. Required\nfields: id/text/surface/group_id/split (strings), label (0/1/null),\nlabel_provenance (human/corpus/synthetic), review_status\n(pending_human_review/human_reviewed/adjudicated/corpus_labeled).\nGroups must encompass work/franchise AND source documents; merge connected groups\nupstream if a source covers multiple works. IDs/groups are globally unique keys,\nnot inferred from titles. Null means uncertain/unlabeled and is never binary gold.\nHuman provenance is an attestation, not something software can independently prove.\n\nHuman-reviewed rows also require reviewer_id and timezone-aware ISO reviewed_at.\nreview_rationale and other additional fields are preserved in the source JSONL.\nHuman policy requires completed review of every row in the target split and uses\nonly human + human_reviewed/adjudicated binary labels. Corpus\npolicy uses only corpus + corpus_labeled binary labels and is explicitly NOT\nproduct gold. Synthetic labels are never scoring targets. Predictions ignore labels.\nAll commands use the exact same complete dataset bytes. Any edit invalidates caches\nand selection. Repeated test evaluation reuses saved predictions; no threshold fit.\n"""\nimport argparse\nfrom datetime import datetime\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport tempfile\n\nVERSION = 1\nSURFACES = {\'headline\', \'comment\', \'review\', \'youtube_transcript\'}\n# Root files only. Never download pickle weights, training_args or nested checkpoints.\nHUB_ALLOW_PATTERNS = [\'config.json\', \'model.safetensors\', \'model.safetensors.index.json\',\n    \'model-?????-of-?????.safetensors\', \'tokenizer.json\', \'tokenizer_config.json\',\n    \'special_tokens_map.json\', \'added_tokens.json\', \'vocab.txt\', \'vocab.json\',\n    \'merges.txt\', \'tokenizer.model\', \'spiece.model\', \'sentencepiece.bpe.model\']\nHUB_IGNORE_PATTERNS = [\'*/*\', \'*.bin\', \'*.pt\', \'*.pth\', \'training_args*\', \'*checkpoint*\']\n\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef atomic_write(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temp = tempfile.mkstemp(prefix=path.name + \'.\', dir=path.parent)\n    try:\n        with os.fdopen(fd, \'w\') as f:\n            json.dump(obj, f, indent=2, allow_nan=False)\n            f.write(\'\\n\')\n            f.flush()\n            os.fsync(f.fileno())\n        os.replace(temp, path)\n    finally:\n        if os.path.exists(temp):\n            os.unlink(temp)\n\n\ndef load_data(path):\n    rows = []\n    ids, groups, texts = set(), {}, {}\n    for line_no, line in enumerate(Path(path).read_text().splitlines(), 1):\n        if not line.strip():\n            continue\n        row = json.loads(line)\n        required = {\'id\', \'text\', \'surface\', \'group_id\', \'split\', \'label\', \'label_provenance\', \'review_status\'}\n        if not isinstance(row, dict) or not required <= row.keys():\n            raise ValueError(f\'Row {line_no}: missing required fields\')\n        for key in (\'id\', \'text\', \'surface\', \'group_id\', \'split\'):\n            if not isinstance(row[key], str) or not row[key].strip():\n                raise ValueError(f\'Row {line_no}: invalid {key}\')\n        if row[\'id\'] in ids:\n            raise ValueError(\'Duplicate id: \' + row[\'id\'])\n        ids.add(row[\'id\'])\n        if row[\'split\'] not in (\'dev\', \'test\') or row[\'surface\'] not in SURFACES:\n            raise ValueError(\'Unsupported split or surface\')\n        if row[\'label\'] is not None and (type(row[\'label\']) is not int or row[\'label\'] not in (0, 1)):\n            raise ValueError(\'label must be integer 0, 1 or null\')\n        if row[\'label_provenance\'] not in (\'human\', \'corpus\', \'synthetic\'):\n            raise ValueError(\'Unsupported label_provenance\')\n        if row[\'review_status\'] not in (\'pending_human_review\', \'human_reviewed\', \'adjudicated\', \'corpus_labeled\'):\n            raise ValueError(\'Unsupported review_status\')\n        reviewed = row[\'review_status\'] in (\'human_reviewed\', \'adjudicated\')\n        synthetic = row[\'label_provenance\'] == \'synthetic\' or row.get(\'is_synthetic\', False) or row.get(\'source_type\') in (\'synthetic\', \'assistant_authored_synthetic\')\n        if reviewed and (row[\'label_provenance\'] != \'human\' or synthetic):\n            raise ValueError(\'Synthetic/nonhuman provenance cannot be passed off as human-reviewed gold\')\n        if reviewed:\n            if not isinstance(row.get(\'reviewer_id\'), str) or not row[\'reviewer_id\'].strip():\n                raise ValueError(\'Human review requires nonempty reviewer_id\')\n            try:\n                reviewed_at = datetime.fromisoformat(row.get(\'reviewed_at\', \'\').replace(\'Z\', \'+00:00\'))\n                if reviewed_at.tzinfo is None or reviewed_at.utcoffset() is None:\n                    raise ValueError(\'Missing time zone\')\n            except (ValueError, TypeError, AttributeError):\n                raise ValueError(\'Human review requires timezone-aware ISO reviewed_at\') from None\n        if row[\'review_status\'] == \'corpus_labeled\' and row[\'label_provenance\'] != \'corpus\':\n            raise ValueError(\'corpus_labeled requires corpus provenance\')\n        if groups.setdefault(row[\'group_id\'], row[\'split\']) != row[\'split\']:\n            raise ValueError(\'Group leakage across dev/test: \' + row[\'group_id\'])\n        normalized = \' \'.join(row[\'text\'].casefold().split())\n        if texts.setdefault(normalized, row[\'split\']) != row[\'split\']:\n            raise ValueError(\'Exact normalized text leakage across dev/test\')\n        rows.append(row)\n    if {r[\'split\'] for r in rows} != {\'dev\', \'test\'}:\n        raise ValueError(\'Provide the complete dataset containing both dev and test\')\n    return rows, file_hash(path)\n\n\ndef eligible(row, policy):\n    if row[\'label\'] is None:\n        return False\n    if policy == \'human\':\n        return row[\'label_provenance\'] == \'human\' and row[\'review_status\'] in (\'human_reviewed\', \'adjudicated\') and not row.get(\'is_synthetic\', False) and row.get(\'source_type\') not in (\'synthetic\', \'assistant_authored_synthetic\')\n    if policy == \'corpus\':\n        return row[\'label_provenance\'] == \'corpus\' and row[\'review_status\'] == \'corpus_labeled\'\n    raise ValueError(\'Unknown label policy\')\n\n\ndef require_split_reviews(rows, split, policy):\n    """Uncertain is reviewed but not binary gold; pending test rows close the gate."""\n    for row in rows:\n        if row[\'split\'] != split:\n            continue\n        if policy == \'human\' and (\'group_review_status\' in row or \'eligible_for_evaluation\' in row):\n            if row.get(\'group_review_status\') != \'verified\' or row.get(\'eligible_for_evaluation\') is not True:\n                raise ValueError(f\'{split} is closed: source/work group verification is pending\')\n        probe = dict(row, label=0)\n        if not eligible(probe, policy):\n            raise ValueError(f\'{split} is closed: every {split} row must complete review under the selected label policy\')\n\n\ndef window_ranges(n_tokens, capacity, overlap, mode):\n    """Half-open content-token spans; overlap is overlap, not advance distance."""\n    if n_tokens < 0 or capacity < 1 or not 0 <= overlap < capacity or mode not in (\'truncate\', \'window\'):\n        raise ValueError(\'Invalid window configuration\')\n    if mode == \'truncate\' or n_tokens <= capacity:\n        return [(0, min(n_tokens, capacity))]\n    spans = []\n    start = 0\n    while True:\n        end = min(start + capacity, n_tokens)\n        spans.append((start, end))\n        if end == n_tokens:\n            return spans\n        start = end - overlap\n\n\ndef aggregate(scores):\n    if not scores or any(not math.isfinite(s) or not 0 <= s <= 1 for s in scores):\n        raise ValueError(\'Expected nonempty finite window scores in [0,1]\')\n    return max(scores)\n\n\ndef metrics(labels, scores, threshold):\n    if len(labels) != len(scores) or any(type(y) is not int or y not in (0, 1) for y in labels):\n        raise ValueError(\'Invalid labels/scores\')\n    if not math.isfinite(threshold):\n        raise ValueError(\'Invalid threshold\')\n    if scores:\n        aggregate(scores)\n    tp = sum(y == 1 and s >= threshold for y, s in zip(labels, scores))\n    fp = sum(y == 0 and s >= threshold for y, s in zip(labels, scores))\n    fn = sum(y == 1 and s < threshold for y, s in zip(labels, scores))\n    tn = len(labels) - tp - fp - fn\n    return dict(n=len(labels), tp=tp, fp=fp, fn=fn, tn=tn,\n                precision=tp/(tp+fp) if tp+fp else 0., recall=tp/(tp+fn) if tp+fn else 0.,\n                f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.)\n\n\ndef select_threshold(labels, scores):\n    """O(n log n); exact tied-score groups, >= rule. F1 ties pick higher cutoff.\n\n    Candidates include all-positive (0), each observed score, and all-negative\n    (nextafter(1,+inf)), so even scores equal to 0 or 1 are handled correctly.\n    """\n    metrics(labels, scores, 0.)\n    if not labels or set(labels) != {0, 1}:\n        raise ValueError(\'Threshold selection requires both binary classes\')\n    total_pos = sum(labels)\n    tp = fp = 0\n    best = (0., math.nextafter(1., math.inf))\n    pairs = sorted(zip(scores, labels), reverse=True)\n    i = 0\n    while i < len(pairs):\n        score = pairs[i][0]\n        while i < len(pairs) and pairs[i][0] == score:\n            tp += pairs[i][1]\n            fp += 1-pairs[i][1]\n            i += 1\n        f1 = 2*tp/(total_pos+tp+fp)\n        best = max(best, (f1, score))\n    best = max(best, (2*total_pos/(total_pos+len(labels)), 0.))\n    return best[1], metrics(labels, scores, best[1])\n\n\ndef seal(obj):\n    obj = dict(obj)\n    obj[\'integrity_sha256\'] = digest(obj)\n    return obj\n\n\ndef read_sealed(path):\n    obj = json.loads(Path(path).read_text())\n    claimed = obj.pop(\'integrity_sha256\', None)\n    if claimed != digest(obj):\n        raise ValueError(\'Artifact integrity mismatch: \' + str(path))\n    return obj\n\n\ndef cache_header(dataset_hash, config, split, selection_hash=None):\n    return dict(version=VERSION, dataset_sha256=dataset_hash, config=config,\n                config_sha256=digest(config), split=split, selection_sha256=selection_hash)\n\n\ndef read_cache(path, expected=None):\n    obj = read_sealed(path)\n    header = obj[\'header\']\n    if header[\'version\'] != VERSION or header[\'config_sha256\'] != digest(header[\'config\']):\n        raise ValueError(\'Invalid cache config fingerprint/version\')\n    if expected is not None and header != expected:\n        raise ValueError(\'Cache mismatch: dataset/model/config/split/selection changed; use a new cache path\')\n    return obj\n\n\ndef validate_predictions(cache, rows, complete=True):\n    target = {r[\'id\']: r for r in rows if r[\'split\'] == cache[\'header\'][\'split\']}\n    predictions = cache[\'predictions\']\n    if not set(predictions) <= target.keys() or (complete and set(predictions) != target.keys()):\n        raise ValueError(\'Cache has missing or foreign records\')\n    for rid, pred in predictions.items():\n        if pred[\'text_sha256\'] != digest(target[rid][\'text\']):\n            raise ValueError(\'Prediction text mismatch\')\n        if pred[\'score\'] != aggregate([w[\'score\'] for w in pred[\'windows\']]):\n            raise ValueError(\'Window aggregation mismatch\')\n    return target\n\n\ndef targets(rows, predictions, split, policy):\n    chosen = [r for r in rows if r[\'split\'] == split and eligible(r, policy)]\n    return chosen, [r[\'label\'] for r in chosen], [predictions[r[\'id\']][\'score\'] for r in chosen]\n\n\ndef validate_selection(selection, dataset_hash, config_hash):\n    if selection[\'version\'] != VERSION or selection[\'dataset_sha256\'] != dataset_hash or selection[\'config_sha256\'] != config_hash:\n        raise ValueError(\'Selection dataset/config fingerprint mismatch\')\n    if selection[\'source_split\'] != \'dev\' or selection[\'label_policy\'] not in (\'human\', \'corpus\'):\n        raise ValueError(\'Selection must come from reviewed dev or explicitly selected corpus dev\')\n    if digest(selection[\'config\']) != selection[\'config_sha256\']:\n        raise ValueError(\'Selection config corruption\')\n    if not 0 <= selection[\'threshold\'] <= math.nextafter(1., math.inf):\n        raise ValueError(\'Invalid frozen threshold\')\n\n\ndef resolve_model(args):\n    if Path(args.model).is_dir():\n        folder = Path(args.model).resolve()\n    else:\n        from huggingface_hub import snapshot_download\n        folder = Path(snapshot_download(args.model, revision=args.revision,\n                                       local_files_only=not args.allow_download,\n                                       allow_patterns=HUB_ALLOW_PATTERNS,\n                                       ignore_patterns=HUB_IGNORE_PATTERNS)).resolve()\n    import fnmatch\n    patterns = HUB_ALLOW_PATTERNS + ([\'pytorch_model.bin\', \'pytorch_model.bin.index.json\',\n        \'pytorch_model-?????-of-?????.bin\'] if Path(args.model).is_dir() else [])\n    files = sorted(p for p in folder.iterdir() if p.is_file() and any(fnmatch.fnmatch(p.name, pattern) for pattern in patterns))\n    if not files:\n        raise ValueError(\'No local HF model/tokenizer files\')\n    hashes = {str(p.relative_to(folder)): file_hash(p) for p in files}\n    return folder, hashes\n\n\ndef predict(args):\n    import torch\n    import transformers\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n    rows, dataset_hash = load_data(args.data)\n    if args.require_cuda and args.device != \'cuda\':\n        raise ValueError(\'--require-cuda requires --device cuda\')\n    if args.device == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA requested but unavailable\')\n    if args.device == \'mps\' and not torch.backends.mps.is_available():\n        raise RuntimeError(\'MPS requested but unavailable\')\n    if args.batch_size < 1 or args.threads < 1 or not 3 <= args.max_length <= 512:\n        raise ValueError(\'Invalid batch size/threads/max length (maximum 512)\')\n    if args.split == \'test\' and not args.selection:\n        raise ValueError(\'Test prediction requires a frozen --selection manifest\')\n    folder, hashes = resolve_model(args)\n    config = dict(pipeline_version=VERSION, model_files=hashes,\n                  checkpoint_sha256=file_hash(args.checkpoint) if args.checkpoint else None,\n                  mode=args.mode, max_length=args.max_length, overlap_tokens=args.stride,\n                  aggregation=\'max\', input=\'text_only\', positive_class=args.positive_class,\n                  device=args.device, batch_size=args.batch_size, threads=args.threads,\n                  torch_version=torch.__version__, transformers_version=transformers.__version__,\n                  dtype=\'float32\', trust_remote_code=False)\n    selection = read_sealed(args.selection) if args.selection else None\n    if selection:\n        validate_selection(selection, dataset_hash, digest(config))\n        if args.split == \'test\':\n            require_split_reviews(rows, \'test\', selection[\'label_policy\'])\n    header = cache_header(dataset_hash, config, args.split, digest(selection) if selection else None)\n    cache = read_cache(args.cache, header) if Path(args.cache).exists() else dict(header=header, predictions={})\n    target = validate_predictions(cache, rows, complete=False)\n    if len(cache[\'predictions\']) == len(target):\n        print(f\'Exact cache hit: {len(target)} records; no inference\')\n        return\n    torch.set_num_threads(args.threads)\n    torch.manual_seed(0)\n    tokenizer = AutoTokenizer.from_pretrained(folder, local_files_only=True, use_fast=True, trust_remote_code=False)\n    if not tokenizer.is_fast:\n        raise ValueError(\'Fast tokenizer required to preserve character offsets\')\n    capacity = args.max_length - tokenizer.num_special_tokens_to_add(pair=False)\n    window_ranges(0, capacity, args.stride, args.mode)\n    model = AutoModelForSequenceClassification.from_pretrained(folder, local_files_only=True, trust_remote_code=False, **({\'num_labels\': 2} if args.checkpoint else {}))\n    if args.checkpoint:\n        # Only load a user-supplied trusted local state checkpoint. No optimizer use.\n        checkpoint = torch.load(args.checkpoint, map_location=\'cpu\', weights_only=True)\n        model.load_state_dict(checkpoint[\'model\'], strict=True)\n        del checkpoint\n    if model.config.num_labels != 2 or args.positive_class not in (0, 1):\n        raise ValueError(\'Requires a two-logit classifier and explicit positive-class index\')\n    if args.max_length > getattr(model.config, \'max_position_embeddings\', args.max_length):\n        raise ValueError(\'max-length exceeds model position capacity\')\n    model.float().to(args.device).eval()\n    for rid, row in target.items():\n        if rid in cache[\'predictions\']:\n            continue\n        encoded = tokenizer(row[\'text\'], add_special_tokens=False, truncation=False, return_offsets_mapping=True, verbose=False)\n        tokens, offsets = encoded[\'input_ids\'], encoded[\'offset_mapping\']\n        spans = window_ranges(len(tokens), capacity, args.stride, args.mode)\n        windows = []\n        with torch.inference_mode():\n            for begin in range(0, len(spans), args.batch_size):\n                batch_spans = spans[begin:begin+args.batch_size]\n                inputs = [tokenizer.prepare_for_model(tokens[a:b], add_special_tokens=True, truncation=False, return_attention_mask=True) for a, b in batch_spans]\n                tensors = tokenizer.pad(inputs, padding=True, return_tensors=\'pt\').to(args.device)\n                scores = model(**tensors).logits.softmax(-1)[:, args.positive_class].cpu().tolist()\n                for (a, b), inp, score in zip(batch_spans, inputs, scores):\n                    windows.append(dict(token_start=a, token_end=b,\n                        char_start=offsets[a][0] if a < b else 0, char_end=offsets[b-1][1] if a < b else 0,\n                        input_token_count=len(inp[\'input_ids\']), score=score))\n        cache[\'predictions\'][rid] = dict(text_sha256=digest(row[\'text\']), surface=row[\'surface\'],\n            full_content_tokens=len(tokens), covered_content_tokens=spans[-1][1],\n            dropped_content_tokens=len(tokens)-spans[-1][1], windows=windows,\n            score=aggregate([w[\'score\'] for w in windows]))\n        atomic_write(args.cache, seal(cache))\n        print(f\'{args.split}: {len(cache["predictions"])}/{len(target)} records; {len(windows)} windows\', flush=True)\n\n\ndef select(args):\n    rows, data_hash = load_data(args.data)\n    require_split_reviews(rows, \'dev\', args.label_policy)\n    candidates = []\n    for path in args.cache:\n        cache = read_cache(path)\n        if cache[\'header\'][\'split\'] != \'dev\' or cache[\'header\'][\'dataset_sha256\'] != data_hash:\n            raise ValueError(\'Configuration/threshold selection requires matching dev predictions only\')\n        validate_predictions(cache, rows)\n        chosen, labels, scores = targets(rows, cache[\'predictions\'], \'dev\', args.label_policy)\n        threshold, result = select_threshold(labels, scores)\n        candidates.append(dict(cache=cache, threshold=threshold, metrics=result,\n                               ids=[r[\'id\'] for r in chosen]))\n    # Deterministic config tie break, independent of caller ordering or test outcomes.\n    winner = sorted(candidates, key=lambda c: (-c[\'metrics\'][\'f1\'], c[\'cache\'][\'header\'][\'config_sha256\']))[0]\n    cache = winner[\'cache\']\n    selection = dict(version=VERSION, source_split=\'dev\', dataset_sha256=data_hash,\n        config_sha256=cache[\'header\'][\'config_sha256\'], config=cache[\'header\'][\'config\'],\n        dev_cache_sha256=digest(cache), threshold=winner[\'threshold\'], comparator=\'>=\',\n        objective=\'maximum dev F1; threshold ties highest cutoff; config ties lexicographic config hash\',\n        label_policy=args.label_policy,\n        evidence=\'human-reviewed product dev\' if args.label_policy == \'human\' else \'corpus-label agreement; NOT product gold\',\n        selected_ids=winner[\'ids\'], excluded_count=sum(r[\'split\'] == \'dev\' for r in rows)-len(winner[\'ids\']),\n        metrics=winner[\'metrics\'], candidates=[dict(config_sha256=c[\'cache\'][\'header\'][\'config_sha256\'],\n            dev_cache_sha256=digest(c[\'cache\']), threshold=c[\'threshold\'], metrics=c[\'metrics\'])\n            for c in sorted(candidates, key=lambda c: c[\'cache\'][\'header\'][\'config_sha256\'])])\n    if Path(args.selection).exists():\n        if read_sealed(args.selection) != selection:\n            raise ValueError(\'Refusing to overwrite a different frozen selection; use a new experiment path\')\n    else:\n        atomic_write(args.selection, seal(selection))\n    print(json.dumps(selection, indent=2))\n\n\ndef evaluate(args):\n    rows, data_hash = load_data(args.data)\n    cache, selection = read_cache(args.cache), read_sealed(args.selection)\n    validate_selection(selection, data_hash, cache[\'header\'][\'config_sha256\'])\n    require_split_reviews(rows, \'test\', selection[\'label_policy\'])\n    if cache[\'header\'][\'split\'] != \'test\' or cache[\'header\'][\'dataset_sha256\'] != data_hash or cache[\'header\'][\'selection_sha256\'] != digest(selection):\n        raise ValueError(\'Test predictions must match dataset and frozen selection\')\n    validate_predictions(cache, rows)\n    chosen, labels, scores = targets(rows, cache[\'predictions\'], \'test\', selection[\'label_policy\'])\n    if not chosen:\n        raise ValueError(\'No eligible test labels under frozen label policy\')\n    result = dict(version=VERSION, split=\'test\', threshold=selection[\'threshold\'],\n        threshold_fitted_on=\'dev only\', selection_sha256=digest(selection), dataset_sha256=data_hash,\n        config_sha256=cache[\'header\'][\'config_sha256\'], label_policy=selection[\'label_policy\'],\n        evidence=\'human-reviewed product test\' if selection[\'label_policy\'] == \'human\' else \'corpus-label agreement; NOT product gold\',\n        metrics=metrics(labels, scores, selection[\'threshold\']),\n        excluded_count=sum(r[\'split\'] == \'test\' for r in rows)-len(chosen), per_surface={})\n    for surface in sorted(SURFACES):\n        selected = [r for r in chosen if r[\'surface\'] == surface]\n        result[\'per_surface\'][surface] = metrics([r[\'label\'] for r in selected], [cache[\'predictions\'][r[\'id\']][\'score\'] for r in selected], selection[\'threshold\'])\n    if Path(args.output).exists() and read_sealed(args.output) != result:\n        raise ValueError(\'Refusing to overwrite different locked test results\')\n    atomic_write(args.output, seal(result))\n    print(json.dumps(result, indent=2))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    sub = parser.add_subparsers(dest=\'command\', required=True)\n    p = sub.add_parser(\'predict\', help=\'Offline by default; cache one completed record atomically at a time\')\n    p.add_argument(\'--data\', required=True)\n    p.add_argument(\'--model\', required=True, help=\'Local HF folder or cached Hub model ID\')\n    p.add_argument(\'--revision\', default=\'main\')\n    p.add_argument(\'--allow-download\', action=\'store_true\', help=\'Explicitly allow Hub downloads; may be large\')\n    p.add_argument(\'--checkpoint\', help=\'Trusted local torch checkpoint with model state dict (existing review epoch2)\')\n    p.add_argument(\'--split\', choices=[\'dev\', \'test\'], required=True)\n    p.add_argument(\'--mode\', choices=[\'truncate\', \'window\'], default=\'truncate\')\n    p.add_argument(\'--max-length\', type=int, default=512)\n    p.add_argument(\'--stride\', type=int, default=128, help=\'Number of overlapping content tokens, not step size\')\n    p.add_argument(\'--device\', choices=[\'cpu\', \'cuda\', \'mps\'], default=\'cpu\')\n    p.add_argument(\'--require-cuda\', action=\'store_true\')\n    p.add_argument(\'--positive-class\', type=int, choices=[0, 1], default=1)\n    p.add_argument(\'--batch-size\', type=int, default=8)\n    p.add_argument(\'--threads\', type=int, default=2)\n    p.add_argument(\'--cache\', required=True)\n    p.add_argument(\'--selection\', help=\'Required before test inference; must match exact dataset and config\')\n    p.set_defaults(func=predict)\n    p = sub.add_parser(\'select\', help=\'Freeze max-F1 threshold from eligible dev labels only\')\n    p.add_argument(\'--data\', required=True)\n    p.add_argument(\'--cache\', required=True, action=\'append\', help=\'Repeat to compare configurations on reviewed dev only\')\n    p.add_argument(\'--selection\', required=True)\n    p.add_argument(\'--label-policy\', choices=[\'human\', \'corpus\'], default=\'human\')\n    p.set_defaults(func=select)\n    p = sub.add_parser(\'evaluate\', help=\'Evaluate test with frozen dev selection; never fit threshold\')\n    p.add_argument(\'--data\', required=True)\n    p.add_argument(\'--cache\', required=True)\n    p.add_argument(\'--selection\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.set_defaults(func=evaluate)\n    args = parser.parse_args()\n    args.func(args)\n\n\nif __name__ == \'__main__\':\n    main()\n')


In [ ]:
import json, sys, subprocess, zipfile, shutil
from google.colab import files
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.1','safetensors>=0.4.3'])
import torch
assert torch.cuda.is_available(), 'Connect a GPU runtime first'
print(torch.cuda.get_device_name(0))
sys.path.insert(0,'/content')
import product_eval as pe


Upload reviewed-spoiler-examples.jsonl, and optionally a prior product-eval-cache.zip. Original unreviewed candidates will be rejected. Keep all rows and split/group IDs unchanged during review.

In [ ]:
uploaded=files.upload()
jsonls=[name for name in uploaded if name.endswith('.jsonl')]
assert len(jsonls)==1, 'Upload exactly one complete reviewed JSONL'
DATA='/content/reviewed.jsonl'
Path(DATA).write_bytes(uploaded[jsonls[0]])
OUT=Path('/content/product-eval'); OUT.mkdir(exist_ok=True)
for name, payload in uploaded.items():
    if name.endswith('.zip'):
        import io
        with zipfile.ZipFile(io.BytesIO(payload)) as z:
            for entry in z.infolist():
                target=(OUT/entry.filename).resolve()
                assert target.is_relative_to(OUT.resolve()), 'Unsafe archive path'
            z.extractall(OUT)
rows, data_hash=pe.load_data(DATA)
for split in ('dev','test'):
    pe.require_split_reviews(rows,split,'human')
assert all(r.get('group_review_status')=='verified' and r.get('eligible_for_evaluation') is True for r in rows), 'Verify source/work grouping before evaluation'
print('Human review gate passed:',len(rows),'rows. Dataset hash:',data_hash)


Run development predictions and freeze the winning configuration. These two candidates use the same pretrained weights; only text coverage differs. No model fitting occurs.

In [ ]:
MODEL='Zritze/imdb-spoiler-robertaOrigDatasetLR1'
REVISION='56fee120f8495ccfc5001e3fbd1478656d17001a'
def run(*args):
    subprocess.run([sys.executable,'/content/product_eval.py',*map(str,args)],check=True)
def predict(mode,split,selection=None):
    args=['predict','--data',DATA,'--model',MODEL,'--revision',REVISION,'--allow-download','--device','cuda','--require-cuda','--split',split,'--mode',mode,'--batch-size','8','--cache',str(OUT/f'{split}-{mode}.json')]
    if selection:args+=['--selection',str(selection)]
    run(*args)
for mode in ('truncate','window'):
    predict(mode,'dev')
SELECTION=OUT/'selection.json'
run('select','--data',DATA,'--cache',OUT/'dev-truncate.json','--cache',OUT/'dev-window.json','--selection',SELECTION,'--label-policy','human')
shutil.make_archive('/content/product-eval-cache','zip',OUT)
files.download('/content/product-eval-cache.zip')


Final held-out evaluation. Set RUN_LOCKED_TEST=True only after reviewing the frozen development selection. No threshold changes after viewing test results. Test caches and results cannot be silently overwritten with a different selection.

In [ ]:
RUN_LOCKED_TEST=False
if RUN_LOCKED_TEST:
    selection=pe.read_sealed(SELECTION)
    mode=selection['config']['mode']
    predict(mode,'test',SELECTION)
    run('evaluate','--data',DATA,'--cache',OUT/f'test-{mode}.json','--selection',SELECTION,'--output',OUT/'test-results.json')
    shutil.make_archive('/content/product-eval-cache','zip',OUT)
    files.download('/content/product-eval-cache.zip')
else:
    print('Test remains locked. Review the development selection before enabling this cell.')
